In [3]:
from pathlib import Path
img_path = Path("data/things-eeg2/imgs")

all_paths_train = sorted((img_path / "training_images").rglob("*.jpg"))
all_paths_test = sorted((img_path / "test_images").rglob("*.jpg"))

all_paths_test[:3]

[PosixPath('data/things-eeg2/imgs/test_images/00001_aircraft_carrier/aircraft_carrier_06s.jpg'),
 PosixPath('data/things-eeg2/imgs/test_images/00002_antelope/antelope_01b.jpg'),
 PosixPath('data/things-eeg2/imgs/test_images/00003_backscratcher/backscratcher_01b.jpg')]

In [4]:
from brain_image.model.img_encoder import load_image_encoder, BaseImageEncoder

clip_img_encoder = load_image_encoder("clip_vitl14")
clip_img_encoder.to("cuda")


OptimizedModule(
  (_orig_mod): CLIPImageEncoder(
    (model): CLIPVisionModelWithProjection(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(257, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    

In [ ]:
import torch
from brain_image.data import batch_load_images
import tqdm

@torch.no_grad()
def embed_images(img_paths: list[Path], batch_size: int, encoder: BaseImageEncoder, use_progressbar: bool = True, device="cuda"):
    N, B = len(img_paths), batch_size
    all_embeds = []

    with tqdm.tqdm(total=N, disable=not use_progressbar, desc="Embedding data...") as pbar:
        for i in range(0, N, B):
            imgs = batch_load_images(img_paths[i:i+B]).to(device)
            embeds = encoder.encode(imgs).detach().cpu()
            all_embeds.append(embeds)
            pbar.update(B)

    return torch.cat(all_embeds, dim=0)

B = 512
all_imgs_train = embed_images(all_paths_train, B, clip_img_encoder)


In [6]:
all_imgs_test = embed_images(all_paths_test, B, clip_img_encoder)

Embedding data...: 512it [00:03, 133.15it/s]              


In [ ]:
from torch import nn
from torch import Tensor

from diffusers.models.embeddings import TimestepEmbedding, Timesteps

@dataclass
class DiffusionPriorConfig:
    d_input: int = 768
    d_cond: int = 768
    d_time: int = 256
    depth: int = 5


class SimpleDiffusionPrior(nn.Module):
    def __init__(self, config: DiffusionPriorConfig):
        super().__init__()
        self.config = config

        self.timesteps = Timesteps(
            self.
        )

    def forward(
            self, 
            x: Tensor, # <B, D>
            timestep: torch.LongTensor, # <B>
            conditioning: torch.Tensor | None = None   # <B, E>
        ):




torch.Size([16540, 768])